## Instalação de dependências

Instala e atualiza todas as bibliotecas necessárias:
- **transformers / trl / peft / accelerate** — fine-tuning de LLMs com LoRA
- **bitsandbytes** — quantização 8-bit para reduzir o uso de VRAM
- **datasets** — datasets HuggingFace
- **sentence-transformers** — similaridade semântica na avaliação
- **evaluate / scikit-learn** — métricas e Random Forest (baseline)
- **pandas / openpyxl / pyreadstat** — leitura de dados (`.SAV`, `.xlsx`, etc.)

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.37.0", "trl>=0.16.0", "peft>=0.13.0", "accelerate>=0.34.0",
    "bitsandbytes>=0.43.0", "datasets>=2.20.0", "sentence-transformers>=3.0.0",
    "evaluate>=0.4.0", "scikit-learn>=1.4.0", "pandas>=2.0.0",
    "openpyxl>=3.1.0", "pyreadstat>=1.3.1"])

0

## Importações e configuração de reprodutibilidade

Importa todas as bibliotecas do projeto e define `SEED = 42` em Python, NumPy, PyTorch e CUDA para garantir resultados reprodutíveis entre execuções.

In [2]:
import math
import os
import random
import re
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 300)

print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA disponível: True
CUDA: NVIDIA GeForce RTX 3080 Ti


## Parâmetros de configuração global

Define todos os hiperparâmetros do experimento em um único lugar:

| Grupo | Parâmetro | Valor |
|---|---|---|
| Dados | `DATA_PATH` | `04829.SAV` |
| Modelo | `BASE_MODEL` | `Qwen/Qwen2.5-0.5B-Instruct` |
| Divisão | `TARGET_TRAIN_FRAC` | 40% |
| Sequência | `MAX_LENGTH` | 256 tokens |
| Treino | `NUM_EPOCHS` / `LEARNING_RATE` | 1 / 2e-4 |
| LoRA | `LORA_R` / `LORA_ALPHA` | 16 / 32 |
| Avaliação | `EVAL_SAMPLE_SIZE` | 30 exemplos |

In [3]:
DATA_PATH = "04829.SAV"

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "/home/rguima1/llm_pesquisa_opiniao/content/qwen_racismo_lora"

TARGET_TRAIN_FRAC = 0.40
MIN_TRAIN_SAMPLES = 2000

MAX_LENGTH = 256
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

MAX_NEW_TOKENS = 30
EVAL_SAMPLE_SIZE = 200

SYSTEM_PROMPT = (
    "Você simula um respondente brasileiro em uma pesquisa sobre racismo. "
    "Responda SOMENTE com o texto exato de uma das opções. Nada mais."
)

## Funções utilitárias de I/O e normalização de texto

- **`load_table(path)`** — carrega o arquivo de dados (`.SAV`, `.csv`, `.xlsx`, `.parquet`, `.json`) de forma genérica, tentando `pd.read_spss` antes de `pyreadstat` como fallback
- **`normalize_text(x)`** — remove espaços extras e caracteres especiais (`\xa0`), padronizando todos os textos antes de usar em prompts ou comparações
- **`clean_generated_text(x)`** — remove tokens inválidos de textos gerados pelo LLM, mantendo apenas caracteres alfanuméricos e pontuação válida

In [4]:
def load_table(path: str) -> pd.DataFrame:
  path_obj = Path(path)
  if not path_obj.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {path}")

  suffix = path_obj.suffix.lower()

  if suffix == ".csv":
    return pd.read_csv(path_obj)

  if suffix in {".xlsx", ".xls"}:
    return pd.read_excel(path_obj)

  if suffix == ".parquet":
    return pd.read_parquet(path_obj)

  if suffix == ".json":
    return pd.read_json(path_obj)

  if suffix == ".sav":
    try:
      return pd.read_spss(path_obj, convert_categoricals=False)
    except Exception:
      import pyreadstat
      df_sav, meta = pyreadstat.read_sav(str(path_obj), apply_value_formats=False)
      print("Leitura via pyreadstat.")
      print("Quantidade de variáveis com labels:", len(getattr(meta, "variable_value_labels", {}) or {}))
      return df_sav

  raise ValueError(f"Formato de arquivo não suportado: {suffix}")

def normalize_text(x) -> str:
  if pd.isna(x):
    return ""
  x = str(x)
  x = x.replace("\xa0"," ")
  x = " ".join(x.split())
  return x.strip()

def clean_generated_text(text: str) -> str:
  text = normalize_text(text)
  text = re.sub(r"[^0-9A-Za-zÀ-ÿ.,;:!?()'\"%/\- ]+", " ", text)
  text = " ".join(text.split())
  return text.strip()

## Carregamento e exploração dos dados brutos

Lê o arquivo `.SAV` (SPSS) com os dados da pesquisa de opinião sobre racismo no Brasil.

**Saída:** `df_raw` com 2 000 respondentes × 144 variáveis.

In [5]:
print("Arquivo existe?", Path(DATA_PATH).exists())
print("Caminho:", DATA_PATH)

df_raw = load_table(DATA_PATH)
print("Formato original:", df_raw.shape)
print("Total de colunas:", len(df_raw.columns))
print(df_raw.columns.tolist())

df_raw.head()

Arquivo existe? True
Caminho: 04829.SAV
Formato original: (2000, 144)
Total de colunas: 144
['SEXO', 'IDADE', 'FX_ID', 'RACA', 'ESCOLARIDADE', 'P1', 'P2', 'P3', 'P4', 'P5A', 'P5B', 'P5C', 'P5D', 'P5E', 'P5F', 'P5G', 'P5H', 'P5I', 'P6_1', 'P6_2', 'P6_3', 'P6_4', 'P6_5', 'P6_6', 'P6_7', 'P6_8', 'P6_9', 'P6_10', 'P7_1', 'P7_2', 'P7_3', 'P7_4', 'P7_5', 'P7_6', 'P8_1', 'P8_2', 'P8_3', 'P8_4', 'P8_5', 'P8_6', 'P8_7', 'P8_8', 'P8_9', 'P8_10', 'P8_11', 'P8_12', 'P8_13', 'P9_1', 'P9_2', 'P9_3', 'P10', 'P11', 'P12', 'P13_1', 'P13_2', 'P13_3', 'P13_4', 'P13_5', 'P13_6', 'P13_7', 'P13_8', 'P14_1', 'P14_2', 'P14_3', 'P14_4', 'P14_5', 'P14_6', 'P14_7', 'P15A', 'P15B', 'P15C', 'P15D', 'P15E', 'P15F', 'P15G', 'P16_1', 'P16_2', 'P16_3', 'P16_4', 'P16_5', 'P16_6', 'P16_7', 'P17_1', 'P17_2', 'P17_3', 'P17_4', 'P17_5', 'P18_1', 'P18_2', 'P18_3', 'P18_4', 'P18_5', 'P18_6', 'P19A', 'P19B', 'P19C', 'P19D', 'P19E', 'P20_1', 'P20_2', 'P20_3', 'P20_4', 'P20_5', 'P20_6', 'P20_7', 'P20_8', 'P20_9', 'P21A', 'P21B'

,SEXO,IDADE,FX_ID,RACA,ESCOLARIDADE,P1,P2,P3,P4,P5A,...,RELIGIAO,REND1,REND2,UF,MUNI,REGIAO,COND,PORTE,ID_Ipec,DATA_ENTREVISTA
0,2.0,43.0,3.0,3.0,14.0,7.0,2.0,2.0,1.0,2.0,...,2.0,6.0,6.0,26.0,2600401.0,2.0,3.0,4.0,182943340.0,2023-04-14
1,2.0,51.0,4.0,3.0,14.0,1.0,2.0,2.0,1.0,2.0,...,1.0,6.0,6.0,26.0,2600401.0,2.0,3.0,4.0,182944002.0,2023-04-14
2,2.0,21.0,1.0,1.0,15.0,5.0,2.0,2.0,1.0,2.0,...,10.0,6.0,5.0,26.0,2600401.0,2.0,3.0,4.0,182944853.0,2023-04-14
3,1.0,63.0,5.0,3.0,7.0,4.0,3.0,2.0,3.0,5.0,...,2.0,6.0,5.0,26.0,2600401.0,2.0,3.0,4.0,182945614.0,2023-04-14
4,2.0,39.0,3.0,1.0,15.0,1.0,99.0,4.0,3.0,1.0,...,10.0,5.0,4.0,43.0,4318903.0,4.0,3.0,4.0,182946145.0,2023-04-14


## Mapeamentos de atributos de perfil dos respondentes

Define `PROFILE_COLS` (lista ordenada de colunas usadas como features do perfil sociodemográfico) e `PROFILE_MAPS` (dicionário que converte o código numérico SPSS → texto descritivo para cada atributo). Esses mapeamentos são usados por `build_profile_text()` para construir a string de perfil inserida no prompt do LLM.

In [6]:
PROFILE_COLS = [
    "SEXO",
    "IDADE",
    "FX_ID",
    "RACA",
    "ESCOLARIDADE",
    "REGIAO",
    "COND",
    "PORT",
    "REND1",
    "REND2",
    "RELIGIAO",
    "P_ORIENT",
    "P_POLITICA",
]

PROFILE_MAPS = {
    "SEXO": {
        1.0: "masculino",
        2.0: "feminino",
    },
    "FX_ID": {
        1.0: "16 a 24 anos",
        2.0: "25 a 34 anos",
        3.0: "35 a 44 anos",
        4.0: "45 a 59 anos",
        5.0: "60 anos ou mais",
    },
    "RACA": {
        1.0: "branca",
        2.0: "preta",
        3.0: "amarela",
        4.0: "parda",
        5.0: "indigena",
    },
    "ESCOLARIDADE": {
        1.0: "analfabeto",
        2.0: "sabe ler e escrever, mas não cursou escola",
        3.0: "pré-escola",
        4.0: "primeira série",
        5.0: "segunda série",
        6.0: "terceira série",
        7.0: "quarta série",
        8.0: "quinta série",
        9.0: "sexta série",
        10.0: "sétima série",
        11.0: "oitava série",
        12.0: "primeiro grau",
        13.0: "segundo grau",
        14.0: "terceiro grau",
        15.0: "superior incompleto",
        16.0: "superior completo",
    },
    "REGIAO": {
        1.0: "Norte",
        2.0: "Nordeste",
        3.0: "Sudeste",
        4.0: "Sul",
        5.0: "Centro-Oeste",
    },
    "COND": {
        1.0: "capital",
        2.0: "periferia",
        3.0: "interior",
    },
    "PORT": {
        1.0: "até 5 mil habitantes",
        2.0: "de 5001 a 10 mil habitantes",
        3.0: "de 10001 a 20 mil habitantes",
        4.0: "de 20001 a 50 mil habitantes",
        5.0: "de 50001 a 100 mil habitantes",
        6.0: "de 100001 a 500 mil habitantes",
        7.0: "mais de 500 mil habitantes",
    },
    "P_ORIENT": {
        1.0: "heterossexual",
        2.0: "homossexual",
        3.0: "bissexual",
        4.0: "assexual",
        5.0: "pansexual",
        6.0: "outra",
        99.0: "não respondeu",
    },
    "P_POLITICA": {
        1.0: "mais à esquerda",
        2.0: "centro",
        3.0: "mais à direita",
        4.0: "não tem posicionamento político",
        99.0: "não respondeu",
    },
    "REND1": {
        1.0: "mais de 20 salários mínimos",
        2.0: "de 10 a 20 salários mínimos",
        3.0: "de 5 a 10 salários mínimos",
        4.0: "de 2 a 5 salários mínimos",
        5.0: "de 1 a 2 salários mínimos",
        6.0: "até 1 salário mínimo",
        98.0: "não tem rendimento pessoal",
        99.0: "não respondeu",
    },
    "REND2": {
        1.0: "mais de 20 salários mínimos",
        2.0: "de 10 a 20 salários mínimos",
        3.0: "de 5 a 10 salários mínimos",
        4.0: "de 2 a 5 salários mínimos",
        5.0: "de 1 a 2 salários mínimos",
        6.0: "até 1 salário mínimo",
        99.0: "não respondeu",
    },
}

## Escalas de resposta e perguntas de escolha única

Define as escalas Likert (`LIKERT_5_99`, `LIKERT_4_99`, etc.) e o dicionário `SINGLE_CHOICE_QUESTIONS`, que mapeia cada código de pergunta (`P1`, `P2`, `P5A`–`P5I`, `P10`, `P11`…) para seu enunciado completo e o conjunto de respostas válidas. Este dicionário é a fonte primária de opções usadas nos prompts do LLM e na validação das respostas.

In [7]:
LIKERT_5_99 = {
    1.0: "Concordo totalmente",
    2.0: "Concordo em parte",
    3.0: "Nem concordo, nem discordo",
    4.0: "Discordo em parte",
    5.0: "Discordo totalmente",
    99.0: "Não se aplica/Não respondeu",
}

LIKERT_5_98 = {
    1.0: "Concordo totalmente",
    2.0: "Concordo em parte",
    3.0: "Nem concordo, nem discordo",
    4.0: "Discordo em parte",
    5.0: "Discordo totalmente",
    98.0: "Não sabe/Não respondeu",
}

AFAVOR_CONTRA = {
    1.0: "A favor",
    2.0: "Contra",
    99.0: "Não se aplica/Não respondeu",
}

EXISTE_NAO_EXISTE = {
    1.0: "Existe",
    2.0: "Não existe",
    99.0: "Não se aplica/Não respondeu",
}

ADEQUACAO = {
    1.0: "Muito adequada",
    2.0: "Pouco adequada",
    3.0: "Nada adequada",
    99.0: "Não sabe/Não respondeu",
    0.0: "",
}

SINGLE_CHOICE_QUESTIONS = {
    "P1": {
        "question": "Dentre as questões listadas a seguir, na sua opinião, qual é a que mais contribui para gerar as desigualdades no Brasil?",
        "answers": {
            1.0: "Raça/Cor/Etnia",
            2.0: "Classe social",
            3.0: "Gênero ou sexo",
            4.0: "Local de moradia",
            5.0: "Deficiência",
            6.0: "Orientação sexual",
            7.0: "Local de origem",
            97.0: "Nenhuma destas/Outras",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P2": {
        "question": "Como você se sente ao responder qual é a sua raça/cor/etnia?",
        "answers": {
            1.0: "Muito confortável",
            2.0: "Confortável",
            3.0: "Desconfortável",
            4.0: "Muito desconfortável",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P3": {
        "question": "Para você definir sua raça/cor/etnia é muito fácil, fácil, difícil ou difícil?",
        "answers": {
            1.0: "Muito fácil",
            2.0: "Fácil",
            3.0: "Difícil",
            4.0: "Muito difícil",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P4": {
        "question": "Na sua opinião, declarar a sua raça/cor/etnia é muito, pouco ou nada importante?",
        "answers": {
            1.0: "Muito importante",
            2.0: "Pouco importante",
            3.0: "Nada importante",
            99.0: "Não sabe/Não respondeu",
        },
    },

    "P5A": {"question": "O Brasil é um país racista.", "answers": LIKERT_5_99},
    "P5B": {"question": "Eu sofro ou já sofri racismo.", "answers": LIKERT_5_99},
    "P5C": {"question": "Eu tenho algumas atitudes e práticas consideradas racistas.", "answers": LIKERT_5_99},
    "P5D": {"question": "Já presenciei situações em que uma pessoa sofreu racismo.", "answers": LIKERT_5_99},
    "P5E": {"question": "Eu trabalho em uma instituição ou empresa racista.", "answers": LIKERT_5_99},
    "P5F": {"question": "Eu estudo em uma escola, faculdade ou universidade racista.", "answers": LIKERT_5_99},
    "P5G": {"question": "Eu convivo com pessoas que sofrem racismo.", "answers": LIKERT_5_99},
    "P5H": {"question": "Eu convivo com pessoas que têm atitudes racistas.", "answers": LIKERT_5_99},
    "P5I": {"question": "Minha família é racista.", "answers": LIKERT_5_99},

    "P10": {
        "question": "Você concorda ou discorda com a criminalização do racismo no Brasil?",
        "answers": {
            1.0: "Concordo totalmente",
            2.0: "Concordo em parte",
            3.0: "Nem concordo, nem discordo",
            4.0: "Discordo em parte",
            5.0: "Discordo totalmente",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P11": {
        "question": "As leis que criminalizam o racismo no Brasil são suficientes?",
        "answers": {
            1.0: "Suficientes apenas para evitar práticas racistas por parte das pessoas",
            2.0: "Suficientes apenas para evitar práticas racistas por parte das instituições ou empresas",
            3.0: "Suficientes para evitar práticas racistas tanto por parte das pessoas quanto das instituições ou empresas",
            4.0: "Não são suficientes para evitar práticas racistas",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P12": {
        "question": "Você sabe ou já ouviu falar sobre racismo ambiental?",
        "answers": {
            1.0: "Sim",
            2.0: "Não",
        },
    },

    "P15A": {"question": "O racismo foi abordado de forma adequada na escola.", "answers": ADEQUACAO},
    "P15B": {"question": "A história e cultura afro-brasileira foi abordada de forma adequada na escola.", "answers": ADEQUACAO},
    "P15C": {"question": "A história e cultura indígena foi abordada de forma adequada na escola.", "answers": ADEQUACAO},
    "P15D": {"question": "A história e cultura africana foi abordada de forma adequada na escola.", "answers": ADEQUACAO},
    "P15E": {"question": "A história das contribuições e do protagonismo das mulheres foi abordada de forma adequada na escola.", "answers": ADEQUACAO},
    "P15F": {"question": "O tema de gênero foi abordado de forma adequada na escola.", "answers": ADEQUACAO},
    "P15G": {"question": "O tema sexualidade foi abordado de forma adequada na escola.", "answers": ADEQUACAO},

    "P19A": {"question": "Pessoas brancas e pessoas negras são tratadas de forma diferente pelas polícias.", "answers": LIKERT_5_98},
    "P19B": {"question": "Pessoas negras são mais criminalizadas e punidas do que as pessoas brancas.", "answers": LIKERT_5_98},
    "P19C": {"question": "A abordagem policial é baseada na cor da pele, tipo de cabelo e tipo de vestimenta das pessoas.", "answers": LIKERT_5_98},
    "P19D": {"question": "O Brasil possui políticas públicas suficientes para garantir inclusão e mais oportunidades para pessoas negras.", "answers": LIKERT_5_98},
    "P19E": {"question": "Aumentar a representatividade das pessoas negras na política e em cargos de poder contribui para diminuir as desigualdades estruturais.", "answers": LIKERT_5_98},

    "P21A": {"question": "Você é a favor ou contra cotas/ações afirmativas de forma geral?", "answers": AFAVOR_CONTRA},
    "P21B": {"question": "Você é a favor ou contra cotas raciais para negros e indígenas?", "answers": AFAVOR_CONTRA},
    "P21C": {"question": "Você é a favor ou contra cotas sociais para pessoas de baixa renda?", "answers": AFAVOR_CONTRA},
    "P21D": {"question": "Você é a favor ou contra cotas para pessoas com deficiência?", "answers": AFAVOR_CONTRA},
    "P21E": {"question": "Você é a favor ou contra cotas para mulheres?", "answers": AFAVOR_CONTRA},
    "P21F": {"question": "Você é a favor ou contra cotas para pessoas LGBTQIA+?", "answers": AFAVOR_CONTRA},

    "P23": {
        "question": "O quanto é importante a convivência entre pessoas com deficiência e pessoas sem deficiência em espaços como escolas, faculdades, universidades e trabalho?",
        "answers": {
            1.0: "Muito importante",
            2.0: "Pouco importante",
            3.0: "Nada importante",
            99.0: "Não sabe/Não respondeu",
        },
    },

    "P24A": {"question": "Na sua cidade, existe acessibilidade em ruas e calçadas?", "answers": EXISTE_NAO_EXISTE},
    "P24B": {"question": "Na sua cidade, existe acessibilidade no transporte público?", "answers": EXISTE_NAO_EXISTE},
    "P24C": {"question": "Na sua cidade, existe acessibilidade em escola/faculdade?", "answers": EXISTE_NAO_EXISTE},
    "P24D": {"question": "Na sua cidade, existe acessibilidade no trabalho?", "answers": EXISTE_NAO_EXISTE},

    "P25": {
        "question": "Você sabe ou já ouviu falar sobre equidade?",
        "answers": {
            1.0: "Sim",
            2.0: "Não",
        },
    },
}

## Perguntas de múltipla seleção e ranking

`MULTI_SELECT_GROUPS` — perguntas onde o respondente pode marcar múltiplas opções (ex.: P6 "Em quais espaços você sofreu racismo?"). Cada grupo define as colunas binárias correspondentes e o mapeamento código → texto.

`RANKING_GROUPS` — perguntas onde o respondente ordena opções por preferência/importância (ex.: P9). Cada grupo define as colunas de posição e o mapeamento código → texto.

In [8]:
MULTI_SELECT_GROUPS = {
    "P6": {
        "question": "Em quais desses espaços você sofre ou já sofreu racismo?",
        "columns": ["P6_1", "P6_2", "P6_3", "P6_4", "P6_5", "P6_6", "P6_7", "P6_8", "P6_9", "P6_10"],
        "choices": {
            1.0: "No ambiente familiar",
            2.0: "No trabalho",
            3.0: "Na escola/faculdade/universidade",
            4.0: "No espaço religioso que frequenta",
            5.0: "Em espaços públicos",
            6.0: "Em estabelecimentos comerciais",
            7.0: "No banco",
            8.0: "Em agências/espaços de recrutamento para trabalho",
            9.0: "Na comunidade onde mora",
            10.0: "No transporte público",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P7": {
        "question": "Dentre essas opções, quais você considera racismo?",
        "columns": ["P7_1", "P7_2", "P7_3", "P7_4", "P7_5", "P7_6"],
        "choices": {
            1.0: "Ação ou prática motivada contra um grupo de uma raça/cor/etnia",
            2.0: "Ação ou prática motivada contra um grupo de uma origem social ou territorial",
            3.0: "Ação motivada contra as práticas culturais de um grupo",
            4.0: "Ação ou prática motivada contra a religião de um grupo",
            5.0: "Ação ou prática motivada devido às características de uma pessoa",
            6.0: "Produção de desigualdades a partir das diferenças entre os grupos",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P8": {
        "question": "Na sua opinião, quais situações representam a forma como as pessoas manifestam o racismo?",
        "columns": ["P8_1", "P8_2", "P8_3", "P8_4", "P8_5", "P8_6", "P8_7", "P8_8", "P8_9", "P8_10", "P8_11", "P8_12", "P8_13"],
        "choices": {
            1.0: "Violência verbal, como xingamentos ou ofensas",
            2.0: "Tratamento desigual",
            3.0: "Negação de oportunidades",
            4.0: "Violência física, como agressões",
            5.0: "Exclusão/isolamento/desprezo de um grupo de pessoas",
            6.0: "Intolerância religiosa",
            7.0: "Conflitos interpessoais entre pessoas de um mesmo grupo",
            8.0: "Práticas ou ações que favorecem um determinado grupo de pessoas",
            9.0: "Desigualdade de investimento em diferentes territórios",
            10.0: "Ações e medidas institucionais",
            11.0: "Negação da história, das contribuições e das potências de um grupo",
            12.0: "Ignorar a existência de um grupo",
            13.0: "Pressionar ou constranger alguém para mudar sua aparência",
            97.0: "Nenhuma destas/Outras",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P13": {
        "question": "Na sua opinião, por quais razões se repetem desastres ambientais em determinadas cidades e regiões do país?",
        "columns": ["P13_1", "P13_2", "P13_3", "P13_4", "P13_5", "P13_6", "P13_7", "P13_8"],
        "choices": {
            1.0: "Descaso do poder público",
            2.0: "Permissão inadequada da exploração de territórios",
            3.0: "Ocupação desordenada das cidades",
            4.0: "Falta de planejamento urbano nas periferias",
            5.0: "Falta de garantia do direito à moradia adequada",
            6.0: "Racismo ambiental",
            7.0: "Falta de fiscalização do poder público",
            8.0: "Descumprimento de normas ambientais pelas empresas",
            97.0: "Nenhuma destas",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P14": {
        "question": "Quais dos temas a seguir você aprendeu na escola?",
        "columns": ["P14_1", "P14_2", "P14_3", "P14_4", "P14_5", "P14_6", "P14_7"],
        "choices": {
            1.0: "Racismo",
            2.0: "História e cultura afro-brasileira",
            3.0: "História e cultura indígena",
            4.0: "História e cultura africana",
            5.0: "História das contribuições e do protagonismo das mulheres",
            6.0: "Gênero",
            7.0: "Sexualidade",
            8.0: "Nenhum desses",
            99.0: "Não sabe/Não lembra/Não respondeu",
        },
    },
    "P16": {
        "question": "Dentre esses temas, quais você considera importantes de serem ensinados ou debatidos para a formação nas escolas?",
        "columns": ["P16_1", "P16_2", "P16_3", "P16_4", "P16_5", "P16_6", "P16_7"],
        "choices": {
            1.0: "Racismo",
            2.0: "História e cultura afro-brasileira",
            3.0: "História e cultura indígena",
            4.0: "História e cultura africana",
            5.0: "História das contribuições e do protagonismo das mulheres",
            6.0: "Gênero",
            7.0: "Sexualidade",
            97.0: "Nenhum desses",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P17": {
        "question": "Você já sofreu algum desses tipos de violência na escola?",
        "columns": ["P17_1", "P17_2", "P17_3", "P17_4", "P17_5"],
        "choices": {
            1.0: "Violência física",
            2.0: "Violência psicológica",
            3.0: "Violência/assédio sexual",
            4.0: "Violência patrimonial",
            5.0: "Violência moral",
            97.0: "Não sofreu nenhum desses tipos de violência na escola",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P18": {
        "question": "Na sua opinião, o que motivou a violência que você sofreu na escola?",
        "columns": ["P18_1", "P18_2", "P18_3", "P18_4", "P18_5", "P18_6"],
        "choices": {
            1.0: "Raça/Cor/Etnia",
            2.0: "Classe social",
            3.0: "Gênero ou sexo",
            4.0: "Local de moradia",
            5.0: "Deficiência",
            6.0: "Orientação sexual",
            7.0: "Local de origem",
            8.0: "Religião",
            9.0: "Opinião política",
            10.0: "Aparência física",
            97.0: "Nenhuma destas/Outras",
            99.0: "Não sabe/Não lembra/Não respondeu",
        },
    },
    "P20": {
        "question": "Em quais desses espaços você sente que o seu grupo étnico-racial está representado de forma adequada?",
        "columns": ["P20_1", "P20_2", "P20_3", "P20_4", "P20_5", "P20_6", "P20_7", "P20_8", "P20_9"],
        "choices": {
            1.0: "Política",
            2.0: "Poder executivo",
            3.0: "Poder legislativo",
            4.0: "Poder judiciário",
            5.0: "Organismos internacionais",
            6.0: "Mídia",
            7.0: "Setor empresarial",
            8.0: "Movimentos sociais",
            9.0: "ONGs",
            97.0: "Não sente que estã representado nesses espaços",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P22": {
        "question": "Você já se beneficiou de algum tipo de cota ou ação afirmativa? Quais?",
        "columns": ["P22_1", "P22_2", "P22_3"],
        "choices": {
            1.0: "De acesso à educação",
            2.0: "De acesso ao emprego",
            3.0: "De acesso à moradia",
            4.0: "Outro tipo de cota ou ação afirmativa",
            97.0: "Nunca se beneficiou de políticas de ação afirmativa/reserva de vagas",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P26": {
        "question": "Para quais temas você acredita que o poder público deveria desenvolver ações de políticas públicas?",
        "columns": ["P26_1", "P26_2", "P26_3", "P26_4", "P26_5", "P26_6", "P26_7"],
        "choices": {
            1.0: "Enfrentamento das desigualdades",
            2.0: "Valorização da diversidade",
            3.0: "Promoção de uma educação antirracista",
            4.0: "Promoção da equidade racial",
            5.0: "Promoção da equidade de gênero",
            6.0: "Enfrentamento do racismo ambiental",
            7.0: "Promoção da acessibilidade para pessoas com deficiência",
            97.0: "Nenhum destes",
            99.0: "Não sabe/Não respondeu",
        },
    },
    "P_PCD": {
        "question": "Você, alguém do seu domicílio ou alguém com quem convive possui algum tipo de deficiência?",
        "columns": ["P_PCD_1", "P_PCD_2", "P_PCD_3"],
        "choices": {
            1.0: "Sim, eu mesmo",
            2.0: "Sim, alguém do domicílio",
            3.0: "Sim, alguém que convive ou se relaciona",
            4.0: "Não",
            98.0: "Não sabe"
        },
    },
}

RANKING_GROUPS = {
    "P9": {
        "question": "Na sua opinião, quais grupos de pessoas mais sofrem racismo no Brasil?",
        "columns": ["P9_1", "P9_2", "P9_3"],
        "choices": {
            1.0: "Pretos",
            2.0: "Pardos",
            3.0: "Indígenas",
            4.0: "Quilombolas",
            5.0: "Brancos",
            6.0: "Asiáticos",
            7.0: "Imigrantes latinos",
            8.0: "Imigrantes africanos",
            9.0: "Imigrantes asiáticos",
            10.0: "Imigrantes europeus",
            97.0: "Nenhum destes",
            99.0: "Não sabe/Não respondeu",
        },
    }
}

## Colunas descartadas

Remove metadados administrativos (`MUNI`, `ID_Ipec`, `DATA_ENTREVISTA`) que não contêm informação demográfica ou de resposta útil para o treinamento.

In [9]:
DROP_COLS = ["MUNI", "ID_Ipec", "DATA_ENTREVISTA"]

## Opções válidas por pergunta

`get_question_options(question_id)` retorna a lista de respostas possíveis para perguntas de escolha única, consultando `SINGLE_CHOICE_QUESTIONS`. Usado para popular o campo `options` em cada registro SFT e para validar respostas no pós-processamento.

In [10]:
def get_question_options(question_id: str) -> list[str]:
    if question_id in SINGLE_CHOICE_QUESTIONS:
        return list(SINGLE_CHOICE_QUESTIONS[question_id]["answers"].values())
    return []

## Decodificação de valores e construção do perfil textual

- **`decode_value(col, value)`** — converte o código numérico SPSS → texto descritivo via `PROFILE_MAPS`
- **`build_profile_text(row)`** — gera a string `"sexo: feminino; idade: 29 anos; raça/cor: parda; …"` inserida no prompt como contexto sociodemográfico do respondente
- **`as_float_or_none(x)`** — conversão segura para `float`, usada nas perguntas de múltipla seleção

In [11]:
def decode_value(col: str, value):
  if pd.isna(value):
    return ""

  if col == "IDADE":
    try:
      return str(int(float(value)))
    except Exception:
      return normalize_text(value)

  if col in PROFILE_MAPS:
    try:
      value = float(value)
    except Exception:
      pass
    return PROFILE_MAPS[col].get(value, normalize_text(value))

  return normalize_text(value)

def build_profile_text(row: pd.Series) -> str:
  parts = []

  for col in PROFILE_COLS:
    if col not in row.index:
      continue

    value = decode_value(col, row[col])
    if not value or value in {"0", "0.0"}:
      continue

    if col == "SEXO":
      parts.append(f"sexo: {value}")
    elif col == "IDADE":
      parts.append(f"idade: {value} anos")
    elif col == "FX_ID":
      parts.append(f"faixa etária: {value}")
    elif col == "RACA":
      parts.append(f"raça/cor: {value}")
    elif col == "ESCOLARIDADE":
      parts.append(f"escolaridade: {value}")
    elif col == "REGIAO":
      parts.append(f"região: {value}")
    elif col == "COND":
      parts.append(f"tipo de município: {value}")
    elif col == "PORT":
      parts.append(f"porte do município: {value}")
    elif col == "REND1":
      parts.append(f"renda pessoal: {value}")
    elif col == "REND2":
      parts.append(f"renda familiar: {value}")
    elif col == "RELIGIAO":
      parts.append(f"religião: {value}")
    elif col == "P_ORIENT":
      parts.append(f"orientação sexual: {value}")
    elif col == "P_POLITICA":
      parts.append(f"posicionamento político: {value}")

  return "; ".join(parts)

def as_float_or_none(x):
  try:
    if pd.isna(x):
      return None
    return float(x)
  except Exception:
    return None

## Construtores de registros por tipo de pergunta

- **`build_single_choice_rows(df)`** — uma resposta por (respondente × pergunta de escolha única); filtra "não respondeu"
- **`build_multi_select_rows(df)`** — concatena as opções marcadas em `"op1, op2, op3"`
- **`build_ranking_rows(df)`** — produz `"1º lugar: X, 2º lugar: Y, 3º lugar: Z"`

Todas usam `build_profile_text()` para gerar o contexto sociodemográfico do respondente.

In [12]:
def build_single_choice_rows(df: pd.DataFrame) -> list[dict]:
    rows = []

    for _, row in df.iterrows():

        profile_text = build_profile_text(row)

        for q_id, q_data in SINGLE_CHOICE_QUESTIONS.items():

            if q_id not in row:
                continue

            value = row[q_id]

            if pd.isna(value):
                continue

            answers_map = q_data["answers"]

            if value not in answers_map:
                continue

            answer_text = answers_map[value]

            if "não respondeu" in answer_text.lower():
                continue

            rows.append({
                "question_id": q_id,
                "question": q_data["question"],
                "answer": answer_text,
                "profile_text": profile_text,
            })

    return rows

def build_multi_select_rows(df: pd.DataFrame) -> List[Dict]:
  rows = []

  for _, row in df.iterrows():
    profile_text = build_profile_text(row)

    for group_name, spec in MULTI_SELECT_GROUPS.items():
      selected = []

      for col in spec["columns"]:
        if col not in row.index:
          continue

        raw_value = as_float_or_none(row.get(col, np.nan))
        if raw_value is None or raw_value == 0.0:
          continue

        decoded = spec["choices"].get(raw_value, "")
        if decoded and decoded not in selected:
          selected.append(decoded)

      if not selected:
        continue

      rows.append({
          "question_group": group_name,
          "question": spec["question"],
          "answer": ", ".join(selected),
          "profile_text": profile_text,
      })

  return rows

def build_ranking_rows(df: pd.DataFrame) -> List[Dict]:
  rows = []

  for _, row in df.iterrows():
    profile_text = build_profile_text(row)

    for group_name, spec in RANKING_GROUPS.items():
      decoded_rank = []

      for idx, col in enumerate(spec["columns"], start=1):
        if col not in df.columns:
          continue

        raw_value = as_float_or_none(row.get(col, np.nan))
        if raw_value is None:
          continue

        label = spec["choices"].get(raw_value, "")
        if not label:
          continue

        decoded_rank.append(f"{idx}º lugar: {label}")

      if not decoded_rank:
        continue

      rows.append({
          "question_group": group_name,
          "question": spec["question"],
          "answer": ", ".join(decoded_rank),
          "profile_text": profile_text,
      })

  return rows

## Dataset no formato longo (*long-format*)

Une os três tipos de pergunta em um único DataFrame, aplica normalização, adiciona a coluna `options`, filtra respostas inválidas (não pertencentes às opções válidas) e deduplica por `(question, answer, profile_text)`.

**Saída:** `df_long` — ~67 000 amostras × 7 colunas.

In [13]:
df = load_table(DATA_PATH).copy()

for col in DROP_COLS:
  if col in df.columns:
    df = df.drop(columns=[col])

print("Formato após remoção de colunas descartadas:", df.shape)

single_rows = build_single_choice_rows(df)
multi_rows = build_multi_select_rows(df)
ranking_rows = build_ranking_rows(df)

df_long = pd.DataFrame(single_rows + multi_rows + ranking_rows)

df_long["options"] = df_long["question_id"].apply(get_question_options)

df_long["question"] = df_long["question"].apply(normalize_text)
df_long["answer"] = df_long["answer"].apply(normalize_text)
df_long["profile_text"] = df_long["profile_text"].apply(normalize_text)

df_long["valid"] = df_long.apply(lambda r: r["answer"] in r["options"], axis=1)

print("Taxa de consistência:", df_long["valid"].mean())

before = len(df_long)

df_long = df_long[
    (df_long["question"] != "") &
    (df_long["answer"] != "") &
    (df_long["valid"] == True)
].copy()

df_long = df_long.drop_duplicates(
    subset=["question", "answer", "profile_text"]
).reset_index(drop=True)

after = len(df_long)

print("Linhas no dataset longo:", before)
print("Linhas após limpeza:", after)
print(df_long.shape)

df_long.head(10)

Formato após remoção de colunas descartadas: (2000, 141)
Taxa de consistência: 0.7447444137003516
Linhas no dataset longo: 90713
Linhas após limpeza: 67452
(67452, 7)


,question_id,question,answer,profile_text,question_group,options,valid
0,P1,"Dentre as questões listadas a seguir, na sua opinião, qual é a que mais contribui para gerar as desigualdades no Brasil?",Local de origem,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Raça/Cor/Etnia, Classe social, Gênero ou sexo, Local de moradia, Deficiência, Orientação sexual, Local de origem, Nenhuma destas/Outras, Não sabe/Não respondeu]",True
1,P2,Como você se sente ao responder qual é a sua raça/cor/etnia?,Confortável,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Muito confortável, Confortável, Desconfortável, Muito desconfortável, Não sabe/Não respondeu]",True
2,P3,"Para você definir sua raça/cor/etnia é muito fácil, fácil, difícil ou difícil?",Fácil,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Muito fácil, Fácil, Difícil, Muito difícil, Não sabe/Não respondeu]",True
3,P4,"Na sua opinião, declarar a sua raça/cor/etnia é muito, pouco ou nada importante?",Muito importante,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Muito importante, Pouco importante, Nada importante, Não sabe/Não respondeu]",True
4,P5A,O Brasil é um país racista.,Concordo em parte,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Concordo totalmente, Concordo em parte, Nem concordo, nem discordo, Discordo em parte, Discordo totalmente, Não se aplica/Não respondeu]",True
5,P5B,Eu sofro ou já sofri racismo.,Discordo totalmente,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Concordo totalmente, Concordo em parte, Nem concordo, nem discordo, Discordo em parte, Discordo totalmente, Não se aplica/Não respondeu]",True
6,P5C,Eu tenho algumas atitudes e práticas consideradas racistas.,Discordo totalmente,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião: 2.0; orientação sexual: heterossexual; posicionamento po...,NaN,"[Concordo totalmente, Concordo em parte, Nem concordo, nem discordo, Discordo em parte, Discordo totalmente, Não se aplica/Não respondeu]",True
7,P5D,Já presenciei situações em que uma pessoa sofreu racismo.,Concordo em parte,sexo: feminino; idade: 43 anos; faixa etária: 35 a 44 anos; raça/cor: amarela; escolaridade: terceiro grau; região: Nordeste; tipo de município: interior; renda pessoal: até 1 salário mínimo; renda familiar: até 1 salário mínimo; religião:

## Distribuição por grupo de pergunta

Contagem de amostras por grupo de pergunta de múltipla seleção e ranking. Perguntas de escolha única têm `question_group = NaN` e são omitidas pelo `value_counts()`.

In [14]:
print("Quantidade de grupo de pergunta:")
df_long["question_group"].value_counts().sort_index()

Quantidade de grupo de pergunta:


Series([], Name: count, dtype: int64)

## Divisão treino / validação / teste

Divide `df_long` em três conjuntos com proporção **40% / 30% / 30%**, com `shuffle=True` e `SEED=42`.

| Conjunto | Amostras |
|---|---|
| Treino | ~26 981 |
| Validação | ~20 235 |
| Teste | ~20 236 |

> Os mesmos conjuntos são reutilizados pelo Random Forest, garantindo comparação justa.

In [15]:
n_total = len(df_long)
n_train = max(MIN_TRAIN_SAMPLES, math.ceil(n_total * TARGET_TRAIN_FRAC))

if n_train >= n_total:
  raise ValueError(f"Dataset pequeno demais: total={n_total}, treino calculado={n_train}")

remaining = n_total - n_train
n_valid = remaining // 2
n_test = remaining - n_valid

print(f"Total: {n_total}")
print(f"Treino: {n_train}")
print(f"Validação: {n_valid}")
print(f"Teste: {n_test}")

train_df, holdout_df = train_test_split(
    df_long,
    train_size=n_train,
    random_state=SEED,
    shuffle=True,
)

valid_ratio = n_valid / len(holdout_df)
valid_df, test_df = train_test_split(
    holdout_df,
    train_size=valid_ratio,
    random_state=SEED,
    shuffle=True,
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Treino:", train_df.shape)
print("Validação:", valid_df.shape)
print("Teste:", test_df.shape)

Total: 67452
Treino: 26981
Validação: 20235
Teste: 20236
Treino: (26981, 7)
Validação: (20235, 7)
Teste: (20236, 7)


## Construção de prompts e registros SFT

- **`build_user_prompt(question, profile_text, options)`** — formata a mensagem do usuário:
  ```
  Perfil: sexo: feminino; idade: 29 anos; …
  Pergunta: O Brasil é um país racista.
  Opções: Concordo totalmente | Concordo em parte | …
  Resposta:
  ```
- **`row_to_sft_record(row)`** — converte uma linha do `df_long` em `{prompt, completion, question, answer, profile_text, options}` para o `SFTTrainer`

In [16]:
def build_user_prompt(question: str, profile_text: str, options: list[str]) -> str:
    parts = []
    if profile_text:
        parts.append(f"Perfil: {profile_text}")
    parts.append(f"Pergunta: {question}")
    if options:
        parts.append("Opções: " + " | ".join(options))
    parts.append("Resposta:")
    return "\n".join(parts)

def row_to_sft_record(row: pd.Series) -> dict:
    question    = normalize_text(row.get("question", ""))
    answer      = normalize_text(row.get("answer", ""))
    profile     = normalize_text(row.get("profile_text", ""))
    options     = row.get("options", [])

    user_content = build_user_prompt(question, profile, options)

    prompt = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": user_content},
    ]
    completion = [{"role": "assistant", "content": answer}]

    return {
        "prompt":       prompt,
        "completion":   completion,
        "question":     question,
        "answer":       answer,
        "profile_text": profile,
        "options":      options,
    }

## Criação dos datasets HuggingFace

Converte os DataFrames de treino, validação e teste em objetos `Dataset` prontos para o `SFTTrainer`. Exibe o schema e um exemplo formatado para verificar a estrutura do prompt.

In [17]:
train_records = [row_to_sft_record(r) for _, r in train_df.iterrows()]
valid_records  = [row_to_sft_record(r) for _, r in valid_df.iterrows()]
test_records   = [row_to_sft_record(r) for _, r in test_df.iterrows()]

train_ds = Dataset.from_list(train_records)
valid_ds  = Dataset.from_list(valid_records)

print(train_ds)
print("\nExemplo formatado:\n")
train_ds[0]

Dataset({
    features: ['prompt', 'completion', 'question', 'answer', 'profile_text', 'options'],
    num_rows: 26981
})

Exemplo formatado:



{'prompt': [{'content': 'Você simula um respondente brasileiro em uma pesquisa sobre racismo. Responda SOMENTE com o texto exato de uma das opções. Nada mais.',
   'role': 'system'},
  {'content': 'Perfil: sexo: masculino; idade: 35 anos; faixa etária: 35 a 44 anos; raça/cor: branca; escolaridade: terceiro grau; região: Norte; tipo de município: interior; renda pessoal: de 1 a 2 salários mínimos; renda familiar: de 1 a 2 salários mínimos; religião: 1.0; orientação sexual: heterossexual; posicionamento político: mais à direita\nPergunta: Para você definir sua raça/cor/etnia é muito fácil, fácil, difícil ou difícil?\nOpções: Muito fácil | Fácil | Difícil | Muito difícil | Não sabe/Não respondeu\nResposta:',
   'role': 'user'}],
 'completion': [{'content': 'Fácil', 'role': 'assistant'}],
 'question': 'Para você definir sua raça/cor/etnia é muito fácil, fácil, difícil ou difícil?',
 'answer': 'Fácil',
 'profile_text': 'sexo: masculino; idade: 35 anos; faixa etária: 35 a 44 anos; raça/cor: 

## Carregamento do modelo base e adapter LoRA

**Modelo base:** `Qwen/Qwen2.5-0.5B-Instruct` com Flash Attention 2 (`bfloat16`).

**LoRA** adiciona matrizes de baixo posto (`r=16`, `alpha=32`) apenas em `q_proj` e `v_proj`, reduzindo os parâmetros treináveis de ~500 M para ~4 M.

> `use_cache=False` é obrigatório durante o treino com `gradient_checkpointing=True`.

In [18]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    attn_implementation="flash_attention_2"
)

model.config.use_cache = False

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"Modelo carregado: {BASE_MODEL}")
print(f"Pad token: {tokenizer.pad_token}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Modelo carregado: Qwen/Qwen2.5-0.5B-Instruct
Pad token: <|endoftext|>


## Fine-tuning com SFTTrainer

Supervised Fine-Tuning com os principais hiperparâmetros:

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `per_device_train_batch_size` | 2 | Evita OOM na T4/A100 |
| `gradient_accumulation_steps` | 8 | Batch efetivo = 16 |
| `packing=True` | — | Elimina padding, acelera treino |
| `lr_scheduler_type` | `cosine` | Decaimento suave da LR |
| `optim` | `paged_adamw_8bit` | Reduz consumo de VRAM |

In [19]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_batch_size = 2 if torch.cuda.is_available() else 1
eval_batch_size = 2 if torch.cuda.is_available() else 1
grad_acc_steps = 8 if torch.cuda.is_available() else 16
optim_name = "paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    gradient_accumulation_steps=grad_acc_steps,
    gradient_checkpointing=True,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="no",
    optim=optim_name,
    bf16=True,
    fp16=False,
    report_to="none",
    max_length=MAX_LENGTH,
    packing=True,
    eos_token="<|im_end|>",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/26981 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/26981 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/20235 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/20235 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
50,0.292173
100,0.215455
150,0.203517
200,0.184357
250,0.188036
300,0.182504
350,0.183644
400,0.178575
450,0.178708
500,0.180235


TrainOutput(global_step=1687, training_loss=0.1823366842965226, metrics={'train_runtime': 3343.0486, 'train_samples_per_second': 8.071, 'train_steps_per_second': 0.505, 'total_flos': 1.262765160336768e+16, 'train_loss': 0.1823366842965226})

## Salvamento do adapter LoRA

Salva os pesos do adapter (~50 MB) e o tokenizer em `OUTPUT_DIR`. Para reutilizar sem re-treinar:

```python
from peft import PeftModel
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, ...)
model = PeftModel.from_pretrained(model, OUTPUT_DIR)
```

In [20]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter LoRA salvo em: {OUTPUT_DIR}")

Adapter LoRA salvo em: /home/rguima1/llm_pesquisa_opiniao/content/qwen_racismo_lora


## Exportação do modelo mesclado (base + LoRA)

Mescla os pesos do adapter com o modelo base e salva o resultado em `OUTPUT_DIR + "_merged"` como modelo autônomo (sem dependência do adapter separado).

> Carregado em CPU (`device_map="cpu"`) para evitar OOM durante a mesclagem.

In [21]:
from peft import PeftModel

MERGED_DIR = OUTPUT_DIR + "_merged"
os.makedirs(MERGED_DIR, exist_ok=True)

base_for_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
)

merged_model = PeftModel.from_pretrained(base_for_merge, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"Modelo mesclado salvo em: {MERGED_DIR}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo mesclado salvo em: /home/rguima1/llm_pesquisa_opiniao/content/qwen_racismo_lora_merged


## Mapeamento de resposta gerada → opção válida

`closest_option(answer, options, embedder)` aplica matching em cascata:

1. **Match exato** (case-insensitive)
2. **Containment** — cobre casos em que o LLM gerou texto extra além da opção
3. **Similaridade semântica** via `SentenceTransformer` (cosseno)
4. **Fallback** — contagem de palavras em comum (sem GPU)

In [22]:
def closest_option(answer: str, options: list[str], embedder=None) -> str:
    if not options:
        return answer

    ans_norm = answer.strip().lower()
    opts_norm = [o.strip().lower() for o in options]

    for opt, opt_norm in zip(options, opts_norm):
        if ans_norm == opt_norm:
            return opt

    for opt, opt_norm in zip(options, opts_norm):
        if opt_norm in ans_norm or ans_norm in opt_norm:
            return opt

    if embedder is not None:
        try:
            vecs = embedder.encode([answer] + options, normalize_embeddings=True)
            sims = vecs[1:] @ vecs[0]
            return options[int(sims.argmax())]
        except Exception:
            pass

    def word_overlap(a: str, b: str) -> int:
        return len(set(a.split()) & set(b.split()))

    scores = [word_overlap(ans_norm, o) for o in opts_norm]
    return options[scores.index(max(scores))]

## Modo de inferência e função de geração

Desabilita o `gradient_checkpointing` e habilita o KV-cache (`use_cache=True`) para geração eficiente.

`generate_answer(question, profile_text, options)`:
1. Monta o prompt com `apply_chat_template()`
2. Tokeniza e move para GPU
3. Gera com **greedy decoding** (`do_sample=False`) e `repetition_penalty=1.3`
4. Decodifica e normaliza apenas os tokens novos

In [23]:
model.config.use_cache = True

if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

model.eval()

device = next(model.parameters()).device

def generate_answer(question: str, profile_text: str, options: list[str]) -> str:
    user_content = build_user_prompt(question, profile_text, options)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return clean_generated_text(raw)

## Sanity check — inferência em um exemplo do teste

Gera e compara a resposta do modelo fine-tunado para o primeiro exemplo do conjunto de teste, verificando que a geração produz respostas coerentes com as opções fornecidas.

In [24]:
sample = test_records[0]
question    = sample["question"]
profile     = sample["profile_text"]
options     = sample["options"]
gold_answer = sample["answer"]

pred_raw = generate_answer(question, profile, options)
pred_mapped = closest_option(pred_raw, options)

print("Pergunta  :", question)
print("Perfil    :", profile)
print("Opções    :", options)
print("Gerada    :", pred_raw)
print("Mapeada   :", pred_mapped)
print("Real      :", gold_answer)
print("Acerto    :", pred_mapped.strip().lower() == gold_answer.strip().lower())

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Pergunta  : O Brasil é um país racista.
Perfil    : sexo: feminino; idade: 52 anos; faixa etária: 45 a 59 anos; raça/cor: amarela; escolaridade: segundo grau; região: Sul; tipo de município: interior; renda pessoal: de 1 a 2 salários mínimos; renda familiar: de 2 a 5 salários mínimos; religião: 1.0; orientação sexual: heterossexual; posicionamento político: centro
Opções    : ['Concordo totalmente', 'Concordo em parte', 'Nem concordo, nem discordo', 'Discordo em parte', 'Discordo totalmente', 'Não se aplica/Não respondeu']
Gerada    : Concordo totalmente
Mapeada   : Concordo totalmente
Real      : Concordo totalmente
Acerto    : True


## Avaliação semântica — utilitários

Carrega o `SentenceTransformer` multilíngue `paraphrase-multilingual-MiniLM-L12-v2` (suporta português) e define:

- **`cosine_similarity_texts(a, b)`** — similaridade de cosseno via embeddings normalizados
- **`evaluate_generation(sample_records)`** — executa inferência em lote e retorna um DataFrame com `gold_answer`, `pred_answer` e `semantic_similarity`

In [25]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def cosine_similarity_texts(texts_a: list[str], texts_b: list[str]) -> list[float]:
    vecs_a = embedder.encode(texts_a, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    vecs_b = embedder.encode(texts_b, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    return (vecs_a * vecs_b).sum(axis=1).tolist()

def evaluate_generation(sample_records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in tqdm(sample_records, desc="Evaluating"):
        question    = rec["question"]
        profile     = rec["profile_text"]
        options     = rec["options"]
        gold        = rec["answer"]

        pred_raw    = generate_answer(question, profile, options)
        pred_mapped = closest_option(pred_raw, options, embedder)
        rows.append({
            "question":          question,
            "profile_text":      profile,
            "options":           options,
            "gold_answer":       gold,
            "pred_answer":       pred_mapped,
        })

    results_df = pd.DataFrame(rows)
    sims = cosine_similarity_texts(
        results_df["gold_answer"].tolist(),
        results_df["pred_answer"].tolist(),
    )
    results_df["semantic_similarity"] = sims
    return results_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Avaliação do LLM — execução em lote

Seleciona `EVAL_SAMPLE_SIZE=30` exemplos aleatórios do conjunto de teste e computa a similaridade semântica entre as respostas geradas e as respostas reais.

In [26]:
random.seed(SEED)
eval_sample = random.sample(test_records, min(EVAL_SAMPLE_SIZE, len(test_records)))

results_df = evaluate_generation(eval_sample)

print(f"Similaridade média: {results_df['semantic_similarity'].mean():.4f}")
print(f"Similaridade mediana: {results_df['semantic_similarity'].median():.4f}")
print(f"Dist. [0.9-1.0]: {(results_df['semantic_similarity'] >= 0.9).sum()} / {len(results_df)}")

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Similaridade média: 0.8238
Similaridade mediana: 1.0000
Dist. [0.9-1.0]: 125 / 200


## Resultados da avaliação semântica

Exibe as primeiras 10 linhas do DataFrame de avaliação com respostas geradas, respostas reais e score de similaridade semântica por par.

In [27]:
results_df[["question", "gold_answer", "pred_answer", "semantic_similarity"]].head(10)

,question,gold_answer,pred_answer,semantic_similarity
0,"Na sua cidade, existe acessibilidade em escola/faculdade?",Existe,Existe,1.000000
1,Você é a favor ou contra cotas sociais para pessoas de baixa renda?,A favor,A favor,1.000000
2,Já presenciei situações em que uma pessoa sofreu racismo.,Discordo totalmente,Concordo totalmente,0.336476
3,Você é a favor ou contra cotas para pessoas LGBTQIA+?,Contra,A favor,0.589785
4,Você é a favor ou contra cotas para pessoas com deficiência?,A favor,A favor,1.000000
5,Eu convivo com pessoas que sofrem racismo.,Concordo totalmente,Discordo totalmente,0.336476
6,Pessoas negras são mais criminalizadas e punidas do que as pessoas brancas.,Concordo em parte,Concordo totalmente,0.857953
7,Aumentar a representatividade das pessoas negras na política e em cargos de poder contribui para diminuir as desigualdades estruturais.,Concordo totalmente,Concordo totalmente,1.000000
8,"Dentre as questões listadas a seguir, na sua opinião, qual é a que mais contribui para gerar as desigualdades no Brasil?",Gênero ou sexo,Raça/Cor/Etnia,0.173204
9,Já presenciei situações em que uma pessoa sofreu racismo.,Concordo totalmente,Concordo totalmente,1.000000


## Inferência ad-hoc com pergunta customizada

Demonstra `generate_answer()` fora do loop de avaliação, passando uma pergunta, perfil sociodemográfico e lista de opções manualmente. As opções de `P5A` (escala Likert 5 pontos) são recuperadas automaticamente de `SINGLE_CHOICE_QUESTIONS`.

In [28]:
nova_pergunta = "O Brasil é um país racista."
novo_perfil = "sexo: feminino; idade: 29 anos; raça/cor: parda; escolaridade: superior completo; região: Sudeste"
novas_opcoes = list(SINGLE_CHOICE_QUESTIONS["P5A"]["answers"].values())

resposta = generate_answer(
    question=nova_pergunta,
    profile_text=novo_perfil,
    options=novas_opcoes,
)

print("RESPOSTA:\n", resposta)

RESPOSTA:
 Concordo totalmente


## Inferência ad-hoc — variante alternativa

Variante do exemplo anterior: recupera a pergunta e as opções diretamente de `SINGLE_CHOICE_QUESTIONS["P5A"]`.

> **Nota:** esta célula é redundante com a anterior e pode ser removida sem impacto no notebook.

In [29]:
nova_pergunta = "O Brasil é um país racista."
novo_perfil = "sexo: feminino; idade: 29 anos; raça/cor: parda; escolaridade: superior completo; região: Sudeste"
novas_opcoes = list(SINGLE_CHOICE_QUESTIONS["P5A"]["answers"].values())

resposta = generate_answer(
    question=nova_pergunta,
    profile_text=novo_perfil,
    options=novas_opcoes,
)

print("RESPOSTA:\n", resposta)

RESPOSTA:
 Concordo totalmente


# Aprendizado Supervisionado — Random Forest

Treinamento de um modelo **Random Forest** usando **TF-IDF** sobre o texto do perfil do participante e a pergunta como features.  
O objetivo é comparar a acurácia deste modelo supervisionado com a do LLM fine-tuned.

In [30]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ---------------------------------------------------------------------------
# 1. Preparação de features
# ---------------------------------------------------------------------------
# Cada amostra recebe como texto: [question_id] + pergunta + perfil
# Isso garante que o RF receba a mesma informação que a LLM.

def build_rf_text(row):
    q_id = str(row["question_id"]) if pd.notna(row.get("question_id")) else ""
    return f"[{q_id}] {row['question']} ||| {row['profile_text']}"

# Codificar rótulos de resposta
le_answer = LabelEncoder()
le_answer.fit(df_long["answer"].values)

y_train_rf = le_answer.transform(train_df["answer"].values)
y_test_rf  = le_answer.transform(test_df["answer"].values)

train_texts_rf = train_df.apply(build_rf_text, axis=1).tolist()
test_texts_rf  = test_df.apply(build_rf_text, axis=1).tolist()

# TF-IDF vetorização
tfidf_rf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train_rf = tfidf_rf.fit_transform(train_texts_rf)
X_test_rf  = tfidf_rf.transform(test_texts_rf)

print(f"Features TF-IDF : {X_train_rf.shape[1]}")
print(f"Classes          : {len(le_answer.classes_)}")
print(f"Treino           : {X_train_rf.shape[0]} amostras")
print(f"Teste            : {X_test_rf.shape[0]} amostras")

# ---------------------------------------------------------------------------
# 2. Treinamento do Random Forest
# ---------------------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
)

print("\nTreinando Random Forest …")
rf_model.fit(X_train_rf, y_train_rf)
print("Treinamento concluído.")

# ---------------------------------------------------------------------------
# 3. Avaliação no conjunto de teste completo
# ---------------------------------------------------------------------------
rf_preds_full   = rf_model.predict(X_test_rf)
rf_labels_full  = le_answer.inverse_transform(rf_preds_full)

rf_acc_full = accuracy_score(y_test_rf, rf_preds_full)
rf_f1_full  = f1_score(y_test_rf, rf_preds_full, average="weighted", zero_division=0)

print(f"\n{'=' * 55}")
print(f"  RF — Acurácia  (teste completo): {rf_acc_full:.4f}")
print(f"  RF — F1 ponder.(teste completo): {rf_f1_full:.4f}")
print(f"{'=' * 55}\n")

print(classification_report(
    le_answer.inverse_transform(y_test_rf),
    rf_labels_full,
    zero_division=0,
))

Features TF-IDF : 955
Classes          : 37
Treino           : 26981 amostras
Teste            : 20236 amostras

Treinando Random Forest …


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    0.5s
[Parallel(n_jobs=-1)]: Done 152 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:    7.0s finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s


Treinamento concluído.


[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.2s
[Parallel(n_jobs=24)]: Done 300 out of 300 | elapsed:    0.4s finished



  RF — Acurácia  (teste completo): 0.6422
  RF — F1 ponder.(teste completo): 0.5827

                                                                                                           precision    recall  f1-score   support

                                                                                                  A favor       0.79      0.96      0.87      2671
                                                                                            Classe social       0.32      0.21      0.25       188
                                                                                        Concordo em parte       0.13      0.01      0.03      1029
                                                                                      Concordo totalmente       0.61      0.78      0.68      3361
                                                                                              Confortável       0.68      0.94      0.79       430
                               

## Comparação: LLM (Qwen 2.5 + LoRA) vs Random Forest

Avalia ambos os modelos na **mesma** amostra aleatória de `EVAL_SAMPLE_SIZE` exemplos do conjunto de teste, usando métricas adequadas por tipo de questão:

| Tipo | Métrica | Critério de "correto" |
|---|---|---|
| **Escolha única** | Exact-match (case-insensitive) | Predição = resposta real |
| **Múltipla seleção / Ranking** | Jaccard sobre elementos separados por vírgula | Score ≥ `JACCARD_THRESHOLD = 0.5` |

Os resultados são exibidos **separados por tipo** e de forma agregada.

In [31]:
def jaccard_score_str(a: str, b: str) -> float:
    """Jaccard similarity on comma-separated element sets."""
    set_a = {x.strip().lower() for x in a.split(",") if x.strip()}
    set_b = {x.strip().lower() for x in b.split(",") if x.strip()}
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

JACCARD_THRESHOLD = 0.5

# --- Identificar tipo de cada exemplo no conjunto de teste ---
is_single   = test_df["question_id"].notna().values
q_type_list = ["single" if f else "multi" for f in is_single]

# --- Amostra comum (todas as questões) ---
comp_size = min(EVAL_SAMPLE_SIZE, len(test_records))
comp_rng  = np.random.default_rng(SEED)
comp_idx  = comp_rng.choice(len(test_records), size=comp_size, replace=False)

comp_records  = [test_records[i] for i in comp_idx]
comp_test_sub = test_df.iloc[comp_idx].reset_index(drop=True)
comp_types    = [q_type_list[i]  for i in comp_idx]

print(f"Amostra: {comp_size} | "
      f"single={sum(t == 'single' for t in comp_types)} | "
      f"multi/ranking={sum(t == 'multi' for t in comp_types)}")

# --- Random Forest na amostra ---
X_comp_rf      = tfidf_rf.transform(comp_test_sub.apply(build_rf_text, axis=1).tolist())
rf_comp_preds  = le_answer.inverse_transform(rf_model.predict(X_comp_rf))

# --- LLM na amostra ---
print(f"Avaliando LLM em {comp_size} amostras …")
rows_comp = []

for i, rec in enumerate(tqdm(comp_records)):
    llm_raw = generate_answer(
        question=rec["question"],
        profile_text=rec.get("profile_text", ""),
        options=rec.get("options", []),
    )
    options = rec.get("options", [])
    gold    = normalize_text(rec["answer"])
    rf_pred = normalize_text(rf_comp_preds[i])
    q_type  = comp_types[i]

    if q_type == "single":
        llm_pred  = closest_option(llm_raw, options, embedder=embedder) if options else normalize_text(llm_raw)
        llm_score = float(llm_pred.lower() == gold.lower())
        rf_score  = float(rf_pred.lower()  == gold.lower())
    else:
        llm_pred  = normalize_text(llm_raw)
        llm_score = jaccard_score_str(llm_pred, gold)
        rf_score  = jaccard_score_str(rf_pred,  gold)

    rows_comp.append({
        "tipo":          q_type,
        "pergunta":      rec["question"][:80],
        "resposta_real": gold,
        "pred_llm":      llm_pred,
        "pred_rf":       rf_pred,
        "llm_score":     round(llm_score, 3),
        "rf_score":      round(rf_score,  3),
        "llm_correto":   llm_score >= JACCARD_THRESHOLD,
        "rf_correto":    rf_score  >= JACCARD_THRESHOLD,
    })

# --- Resultados por tipo de questão ---
df_comp = pd.DataFrame(rows_comp)

print(f"\n{'=' * 65}")
print(f"  {'Tipo':<22} {'LLM':>10} {'RF':>10} {'N':>6}")
print(f"  {'-' * 52}")
for tipo, label in [("single", "Escolha única"), ("multi", "Múltipla/Ranking"), ("all", "Geral")]:
    sub = df_comp if tipo == "all" else df_comp[df_comp["tipo"] == tipo]
    if len(sub) == 0:
        continue
    print(f"  {label:<22} {sub['llm_correto'].mean():>10.4f} {sub['rf_correto'].mean():>10.4f} {len(sub):>6}")
print(f"\n  {'Acurácia RF (teste completo)':<22} {'—':>10} {rf_acc_full:>10.4f}")
print(f"  {'F1 pond. RF (teste completo)':<22} {'—':>10} {rf_f1_full:>10.4f}")
print(f"{'=' * 65}")

# --- Concordância geral ---
both_correct = ((df_comp["llm_correto"]) & (df_comp["rf_correto"])).sum()
only_llm     = ((df_comp["llm_correto"]) & (~df_comp["rf_correto"])).sum()
only_rf      = ((~df_comp["llm_correto"]) & (df_comp["rf_correto"])).sum()
both_wrong   = ((~df_comp["llm_correto"]) & (~df_comp["rf_correto"])).sum()

print(f"\n{'Concordância':<40} {'Qtd':>5}")
print(f"{'-' * 45}")
print(f"{'Ambos acertaram':<40} {both_correct:>5}")
print(f"{'Só LLM acertou':<40} {only_llm:>5}")
print(f"{'Só RF acertou':<40} {only_rf:>5}")
print(f"{'Ambos erraram':<40} {both_wrong:>5}")
print(f"\n--- Detalhes da amostra ---")
df_comp

Amostra: 200 | single=200 | multi/ranking=0
Avaliando LLM em 200 amostras …


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done   2 tasks      | elapsed:    0.0s
[Parallel(n_jobs=24)]: Done 152 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 300 out of 300 | elapsed:    0.1s finished


  0%|          | 0/200 [00:00<?, ?it/s]


  Tipo                          LLM         RF      N
  ----------------------------------------------------
  Escolha única              0.6950     0.6700    200
  Geral                      0.6950     0.6700    200

  Acurácia RF (teste completo)          —     0.6422
  F1 pond. RF (teste completo)          —     0.5827

Concordância                               Qtd
---------------------------------------------
Ambos acertaram                            128
Só LLM acertou                              11
Só RF acertou                                6
Ambos erraram                               55

--- Detalhes da amostra ---


,tipo,pergunta,resposta_real,pred_llm,pred_rf,llm_score,rf_score,llm_correto,rf_correto
0,single,"A abordagem policial é baseada na cor da pele, tipo de cabelo e tipo de vestimen",Concordo em parte,Concordo totalmente,Concordo totalmente,0.0,0.0,False,False
1,single,Você é a favor ou contra cotas sociais para pessoas de baixa renda?,A favor,A favor,A favor,1.0,1.0,True,True
2,single,Você concorda ou discorda com a criminalização do racismo no Brasil?,Concordo totalmente,Concordo totalmente,Concordo totalmente,1.0,1.0,True,True
3,single,Aumentar a representatividade das pessoas negras na política e em cargos de pode,Concordo totalmente,Concordo totalmente,Concordo totalmente,1.0,1.0,True,True
4,single,O Brasil possui políticas públicas suficientes para garantir inclusão e mais opo,Concordo totalmente,Concordo totalmente,Discordo totalmente,1.0,0.0,True,False
...,...,...,...,...,...,...,...,...,...
195,single,"Dentre as questões listadas a seguir, na sua opinião, qual é a que mais contribu",Raça/Cor/Etnia,Raça/Cor/Etnia,Raça/Cor/Etnia,1.0,1.0,True,True
196,single,"A abordagem policial é baseada na cor da pele, tipo de cabelo e tipo de vestimen",Concordo totalmente,Concordo totalmente,Concordo totalmente,1.0,1.0,True,True
197,single,O Brasil é um país racista.,Concordo totalmente,Concordo totalmente,Concordo totalmente,1.0,1.0,True,True
198,single,Eu trabalho em uma instituição ou empresa racista.,Concordo totalmente,Discordo totalmente,Discordo totalmente,0.0,0.0,False,False
